In [1]:
# Import required modules
import cv2
import numpy as np
import os
import glob

# Define the dimensions of checkerboard
CHECKERBOARD = (9, 6)

# stop the iteration when specified
# accuracy, epsilon, is reached or
# specified number of iterations are completed.
criteria = (cv2.TERM_CRITERIA_EPS +
			cv2.TERM_CRITERIA_MAX_ITER, 50, 0.001)

# Vector for 3D points
threedpoints = []

# Vector for 2D points
twodpoints = []

# 3D points real world coordinates
objectp3d = np.zeros((1, CHECKERBOARD[0]
					* CHECKERBOARD[1],
					3), np.float32)
objectp3d[0, :, :2] = np.mgrid[0:CHECKERBOARD[0],
							0:CHECKERBOARD[1]].T.reshape(-1, 2)
prev_img_shape = None

# Extracting path of individual image stored
# in a given directory. Since no path is
# specified, it will take current directory
# jpg files alone
images = glob.glob('./checkerboards/*.jpg')

for i, filename in enumerate(images):
	image = cv2.imread(filename)
	image = cv2.resize(image, (1000, 750), interpolation=cv2.INTER_AREA)
	grayColor = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

	# Find the chess board corners
	# If desired number of corners are
	# found in the image then ret = true
	ret, corners = cv2.findChessboardCorners(
					grayColor, CHECKERBOARD,
					cv2.CALIB_CB_ADAPTIVE_THRESH
					+ cv2.CALIB_CB_FAST_CHECK +
					cv2.CALIB_CB_NORMALIZE_IMAGE)

	# If desired number of corners can be detected then,
	# refine the pixel coordinates and display
	# them on the images of checker board
	if ret == True:
		threedpoints.append(objectp3d)

		# Refining pixel coordinates
		# for given 2d points.
		corners2 = cv2.cornerSubPix(
			grayColor, corners, (20, 20), (-1, -1), criteria)

		twodpoints.append(corners2)

		# Draw and display the corners
		image = cv2.drawChessboardCorners(image,
										CHECKERBOARD,
										corners2, ret)

	# cv2.imshow('img', image)
	cv2.imwrite('./corner/{}.jpg'.format(i), image)
	# cv2.waitKey(0)


h, w = image.shape[:2]

# Perform camera calibration by
# passing the value of above found out 3D points (threedpoints)
# and its corresponding pixel coordinates of the
# detected corners (twodpoints)
ret, matrix, distortion, r_vecs, t_vecs = cv2.calibrateCamera(
	threedpoints, twodpoints, grayColor.shape[::-1], None, None)

# Displaying required output
print("Camera matrix:\n")
print(matrix)

print("Distortion coefficient:\n")
print(distortion)

print("Rotation Vectors:\n")
print(r_vecs)

print("Translation Vectors:\n")
print(t_vecs)

Camera matrix:

[[733.39419536   0.         503.82907971]
 [  0.         732.11725155 369.89539955]
 [  0.           0.           1.        ]]
Distortion coefficient:

[[ 0.24604382 -1.01102813 -0.00450469  0.0026482   1.64885357]]
Rotation Vectors:

(array([[-0.12126394],
       [ 0.13463599],
       [ 3.12597176]]), array([[-0.40451283],
       [ 0.63168547],
       [ 2.07776561]]), array([[ 0.04739598],
       [ 0.64598478],
       [-3.04978469]]), array([[-0.0586642 ],
       [ 0.94653745],
       [ 2.97498748]]), array([[-0.02910526],
       [ 0.01855234],
       [ 3.13606086]]), array([[-0.11132283],
       [-0.73014243],
       [-2.56585068]]))
Translation Vectors:

(array([[4.1860856 ],
       [2.4967591 ],
       [9.54844998]]), array([[ 4.58079159],
       [-2.18317003],
       [13.9370839 ]]), array([[ 3.9458633 ],
       [ 3.89124155],
       [12.09918674]]), array([[ 4.28616412],
       [ 1.85310577],
       [10.23931967]]), array([[4.20451865],
       [2.80178765],
      

In [2]:
# undistort
# samples = glob.glob('./checkerboards/*.jpg')
samples = glob.glob('./samples/*.jpg')
for i, filename in enumerate(samples):
    img = cv2.imread(filename)
    img = cv2.resize(img, (1000, 750), interpolation=cv2.INTER_AREA)
    h, w = img.shape[:2]
    newcameramtx, roi = cv2.getOptimalNewCameraMatrix(matrix, distortion, (w,h), 1, (w,h))
    # undistort
    dst = cv2.undistort(img, matrix, distortion, None, newcameramtx)
    cv2.imwrite('./undistorted/calibresult_nocrop_{}.png'.format(i), dst)
    # crop the image
    x, y, w, h = roi
    dst = dst[y:y+h, x:x+w]
    cv2.imwrite('./undistorted/calibresult_{}.png'.format(i), dst)